# Bootstrap Activity

Fitting `body_mass_g` by `flipper_length_mm`, `bill_length_mm`, and `bill_depth_mm`.

In [ ]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
from plotnine import *
from plotnine.data import penguins as penguins_raw

penguins = penguins_raw.dropna().copy()

## Step By Step

### Set Up

In [ ]:
rng = np.random.default_rng(12345)

bootstrap_predictors = [
    "flipper_length_mm",
    "bill_length_mm",
    "bill_depth_mm",
]

### Fit Model

In [ ]:
X_bootstrap = sm.add_constant(
    penguins[bootstrap_predictors],
    has_constant="add",
)
y_bootstrap = penguins["body_mass_g"]

bootstrap_model = sm.OLS(y_bootstrap, X_bootstrap).fit()

### Draw Indexes

In [ ]:
sample_positions = rng.integers(
    low=0,
    high=len(penguins),
    size=len(penguins),
)

### Create Sample

In [ ]:
bootstrap_sample = penguins.iloc[sample_positions]

### Prep Data

In [ ]:
X_sample = sm.add_constant(
    bootstrap_sample[bootstrap_predictors],
    has_constant="add",
)

y_sample = bootstrap_sample["body_mass_g"]

### Fit Data 

In [ ]:
fitted_sample = sm.OLS(
    endog=y_sample,
    exog=X_sample,
).fit()


### Implementing For Loop

In [ ]:
B = 1000
rng = np.random.default_rng(12345)
bootstrap_estimates = []

for _ in range(B):
    sample_positions = rng.integers(
        low=0,
        high=len(penguins),
        size=len(penguins),
    )
    bootstrap_sample = penguins.iloc[sample_positions]

    X_sample = sm.add_constant(
        bootstrap_sample[bootstrap_predictors],
        has_constant="add",
    )
    y_sample = bootstrap_sample["body_mass_g"]

    fitted_sample = sm.OLS(y_sample, X_sample).fit()
    bootstrap_estimates.append(fitted_sample.params)



### Obtianing Results

In [ ]:
bootstrap_estimates = pd.DataFrame(bootstrap_estimates)

bootstrap_results = pd.DataFrame(
    {
        "estimate": bootstrap_model.params,
        "bootstrap_se": bootstrap_estimates.std(ddof=1),
        "ci_lower": bootstrap_estimates.quantile(0.025),
        "ci_upper": bootstrap_estimates.quantile(0.975),
    }
)

bootstrap_results.round(3)